# Task 5: Edge Deployment (T4 Sim)

Deploy INT4 quantized model on edge-simulating hardware (T4 16GB).
Verify accuracy, measure FPS, profile memory.

## Setup

In [ ]:
import os, sys, json, re, time, torch, gc
import numpy as np
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt

sys.path.insert(0, '../../task4-fine-tuning/granulometry')
from config import *

QUANTIZED_PATH = './quantized_model_int4'  # copy from workbench

print(f'GPU: {torch.cuda.get_device_name(0)} — {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## Load Quantized Model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

print(f'Loading INT4 model from {QUANTIZED_PATH}...')
processor = AutoProcessor.from_pretrained(QUANTIZED_PATH, min_pixels=256*28*28, max_pixels=512*28*28)

# Try AWQ first, fall back to standard
try:
    from awq import AutoAWQForCausalLM
    model = AutoAWQForCausalLM.from_quantized(QUANTIZED_PATH, fuse_layers=True)
    print('Loaded AWQ model')
except:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QUANTIZED_PATH, device_map='auto', torch_dtype=torch.float16)
    print('Loaded standard quantized model')

model_mem = torch.cuda.memory_allocated(0) / 1e9
print(f'Model VRAM: {model_mem:.1f} GB')
print(f'Free VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.1f} GB')


## Eval Helpers

In [ ]:
def parse_response(raw):
    if not raw: return None
    raw = raw.replace('<','').replace('>','')
    raw = re.sub(r'```json\s*','',raw); raw = re.sub(r'```\s*','',raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    sm = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    gm = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if sm and gm: return {'max_particle_size_mm': int(sm.group(1)), 'grading': gm.group(1)}
    return None

with open(TEST_MANIFEST) as f: test_manifest = json.load(f)
quick = [next(e for e in test_manifest if e['class']==cls) for cls in sorted(GT.keys())]
print(f'Test: {len(test_manifest)}, Quick: {len(quick)}')


## Quick Eval (9 images)

In [ ]:
model.eval()
print('Quick eval on edge:')
for entry in quick:
    img_path = os.path.join(TEST_DIR, entry['image'])
    image = Image.open(img_path).convert('RGB')
    scale = min(MAX_DIM/max(image.size), 1.0)
    gsd = ORIGINAL_GSD * scale
    prompt = make_prompt(gsd)
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
    t=time.time()
    with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    elapsed=time.time()-t
    out = processor.batch_decode(ids[:,inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    del inputs, ids; image.close(); torch.cuda.empty_cache()
    parsed = parse_response(out)
    gs=entry['max_particle_size_mm']; gg=entry['grading']
    if parsed:
        ps=parsed.get('max_particle_size_mm','?'); pg=parsed.get('grading','?')
        sv='✓' if ps==gs else '✗'; gv='✓' if pg==gg else '✗'
        print(f'  {entry["class"]:>3} GT:{gs}mm/{gg} Pred:{ps}mm/{pg} {sv}{gv} {elapsed:.1f}s')
    else:
        print(f'  {entry["class"]:>3} PARSE FAIL {elapsed:.1f}s')


## Full Eval (108 images)

In [ ]:
results=[]; cs=0; cg=0; vj=0; tt=0; times=[]
for i, entry in enumerate(test_manifest):
    img_path = os.path.join(TEST_DIR, entry['image'])
    if not os.path.exists(img_path): continue
    image = Image.open(img_path).convert('RGB')
    scale = min(MAX_DIM/max(image.size), 1.0)
    gsd = ORIGINAL_GSD * scale
    prompt = make_prompt(gsd)
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
    t=time.time()
    with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    elapsed=time.time()-t; tt+=elapsed; times.append(elapsed)
    out = processor.batch_decode(ids[:,inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    del inputs, ids; image.close(); torch.cuda.empty_cache()
    parsed = parse_response(out)
    gs=entry['max_particle_size_mm']; gg=entry['grading']; so=False; go=False
    if parsed:
        vj+=1; ps=parsed.get('max_particle_size_mm')
        if isinstance(ps,str): ps=int(ps) if ps.isdigit() else None
        if ps==gs: so=True; cs+=1
        if parsed.get('grading','').lower()==gg: go=True; cg+=1
    results.append({'image':entry['image'],'class':entry['class'],'gt_size':gs,'gt_grading':gg,
        'predicted':parsed,'raw':out,'size_correct':so,'grading_correct':go,'valid_json':parsed is not None,'time_s':round(elapsed,2)})
    if (i+1)%20==0:
        n=i+1; print(f'  [{n}/{len(test_manifest)}] Size:{cs}/{n}({cs/n*100:.0f}%) Grade:{cg}/{n}({cg/n*100:.0f}%)')

n=len(results); both=sum(1 for x in results if x['size_correct'] and x['grading_correct'])
print(f'\nEdge INT4: Size={cs}/{n}({cs/n*100:.1f}%) Grade={cg}/{n}({cg/n*100:.1f}%) Both={both}/{n}({both/n*100:.1f}%)')
print(f'Avg: {tt/n:.2f}s/img | FPS: {n/tt:.2f} | Peak VRAM: {torch.cuda.max_memory_allocated(0)/1e9:.1f} GB')


## FPS Benchmark
Sustained inference speed over 50 consecutive images.

In [ ]:
# Warmup
for _ in range(3):
    entry = test_manifest[0]
    image = Image.open(os.path.join(TEST_DIR, entry['image'])).convert('RGB')
    scale = min(MAX_DIM/max(image.size), 1.0)
    prompt = make_prompt(ORIGINAL_GSD * scale)
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
    with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    del inputs, ids; image.close(); torch.cuda.empty_cache()

# Benchmark
fps_times = []
for i in range(50):
    entry = test_manifest[i % len(test_manifest)]
    image = Image.open(os.path.join(TEST_DIR, entry['image'])).convert('RGB')
    scale = min(MAX_DIM/max(image.size), 1.0)
    prompt = make_prompt(ORIGINAL_GSD * scale)
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
    torch.cuda.synchronize()
    t = time.time()
    with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    torch.cuda.synchronize()
    fps_times.append(time.time() - t)
    del inputs, ids; image.close(); torch.cuda.empty_cache()

avg_fps = 1.0 / np.mean(fps_times)
p50 = np.percentile(fps_times, 50)
p95 = np.percentile(fps_times, 95)
print(f'Sustained FPS: {avg_fps:.2f}')
print(f'Latency — mean: {np.mean(fps_times):.2f}s, p50: {p50:.2f}s, p95: {p95:.2f}s')
print(f'Target: ≥1 FPS → {"PASS" if avg_fps >= 1.0 else "FAIL"}')

plt.figure(figsize=(10, 4))
plt.plot(fps_times, 'o-', markersize=3)
plt.axhline(y=1.0, color='r', linestyle='--', label='1 FPS target')
plt.xlabel('Image #'); plt.ylabel('Time (s)'); plt.title('Edge Inference Latency')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


## Memory Profile

In [ ]:
print(f'Model VRAM: {model_mem:.1f} GB')
print(f'Peak VRAM during inference: {torch.cuda.max_memory_allocated(0)/1e9:.1f} GB')
print(f'Total GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Headroom: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.max_memory_allocated(0))/1e9:.1f} GB')
print(f'\nThis simulates Jetson Orin NX 16GB deployment.')


## Save Results

In [ ]:
with open('results_edge_int4.json','w') as f:
    json.dump({'model':MODEL_ID,'precision':'int4','location':'edge_sim_t4',
        'total_images':n,'json_validity_pct':round(vj/n*100,1),
        'size_accuracy_pct':round(cs/n*100,1),'grading_accuracy_pct':round(cg/n*100,1),
        'both_correct_pct':round(both/n*100,1),'avg_inference_time_s':round(tt/n,2),
        'sustained_fps':round(avg_fps,2),'peak_vram_gb':round(torch.cuda.max_memory_allocated(0)/1e9,1),
        'model_vram_gb':round(model_mem,1),'results':results}, f, indent=2)
print('Saved results_edge_int4.json')
